Inspection of variant RGB images

In [1]:
from variants import SingleVariant, TrioVariant, SingleVariantLRS, TrioVariantLRS
import pysam
import numpy as np
import pandas as pd
import glob

In [2]:
# read the .csv file with predictions
predictions = pd.read_csv('/ifs/home/ina/DeNovoCNN/test/prediction.chr21.csv', sep='\t')
predictions

,Chromosome,Start position,Reference,Variant,DeNovoCNN probability,Child coverage,Father coverage,Mother coverage,Child BAM,Father BAM,Mother BAM
0,chr21,5217060,C,T,0.001,56,68,52,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...
1,chr21,5217116,A,G,0.000,59,69,52,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...
2,chr21,5217224,C,G,0.001,61,69,53,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...
3,chr21,5217336,C,A,0.000,60,69,53,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...
4,chr21,5217337,C,A,0.000,57,68,49,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...
...,...,...,...,...,...,...,...,...,...,...,...
17623,chr21,46699903,A,G,0.000,2,2,0,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...
17624,chr21,46699915,A,G,0.000,2,1,0,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...
17625,chr21,46699921,A,G,0.000,2,2,2,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...
17626,chr21,46699927,A,G,0.000,2,2,2,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...,/ifs/data/research/projects/ina/test_trio/DNA2...


In [5]:
# filter the de novo variants
DNMs = predictions[predictions['DeNovoCNN probability'] >= 0.5]
num_DNMs = DNMs.shape[0]
# DNMs.sample(10)
# print(num_DNMs)          

In [4]:
# DNMs[DNMs['Start position']==7916210]
# DNMs.info()

In [2]:
# read the dataset
dataset = pd.read_csv('/ifs/data/research/projects/gelana/denovocnn_lrs/data/revio_dnm/training_dataset.tsv', sep='\t')
dataset

,chrom,pos,ref,alt,hap_counts,variant_type_x,child,number_of_haplotypes,hap1_var_reads,hap1_total_reads,...,mother_number_of_haplotypes,mother_hap1_var_reads,mother_hap1_total_reads,mother_hap2_var_reads,mother_hap2_total_reads,mother_hap_na_var_reads,mother_hap_na_total_reads,mother_hap1_vaf,mother_hap2_vaf,DNM_status
0,chr1,13252342,C,T,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
1,chr1,13253645,G,C,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
2,chr1,22578139,T,A,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
3,chr1,80330738,A,G,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
4,chr1,101553191,A,T,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36505,chr1,214834227,A,ATTATAC,defaultdict(<function count_variants_by_haplot...,ins,DNA24-12831,2.0,8.0,11.0,...,2.0,0.0,13.0,0.0,9.0,0.0,0.0,0.0,0.000000,UNKNOWN
36506,chr14,25966217,GA,G,defaultdict(<function count_variants_by_haplot...,del,DNA24-12831,2.0,0.0,11.0,...,2.0,0.0,13.0,0.0,17.0,0.0,0.0,0.0,0.000000,UNKNOWN
36507,chr19,37293445,C,A,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,3.0,0.0,8.0,...,3.0,0.0,37.0,6.0,22.0,0.0,18.0,0.0,0.272727,UNKNOWN
36508,chr2,232995222,T,C,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,1.0,18.0,...,2.0,0.0,5.0,1.0,16.0,0.0,0.0,0.0,0.062500,UNKNOWN


In [3]:
# filter possible de novos in the dataset for the sample of interest
possible_DNMs = dataset[(dataset['DNM_status'] == "POSSIBLY_PHASED_DNM") & (dataset['child'] == 'DNA21-19521')]
possible_DNMs

,chrom,pos,ref,alt,hap_counts,variant_type_x,child,number_of_haplotypes,hap1_var_reads,hap1_total_reads,...,mother_number_of_haplotypes,mother_hap1_var_reads,mother_hap1_total_reads,mother_hap2_var_reads,mother_hap2_total_reads,mother_hap_na_var_reads,mother_hap_na_total_reads,mother_hap1_vaf,mother_hap2_vaf,DNM_status
18087,chr1,58778159,G,A,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,9.0,10.0,...,2.0,0.0,6.0,0.0,10.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
18088,chr1,87953395,C,T,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,17.0,...,2.0,0.0,14.0,0.0,12.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
18089,chr1,102136328,A,G,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,14.0,...,2.0,0.0,20.0,0.0,22.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
18090,chr1,163209622,T,C,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,13.0,13.0,...,2.0,0.0,14.0,0.0,14.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
18091,chr1,169369302,T,C,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,15.0,15.0,...,2.0,0.0,25.0,0.0,17.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18176,chr8,75481144,G,C,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,20.0,20.0,...,2.0,0.0,15.0,0.0,15.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
18177,chr8,103265984,A,C,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,15.0,...,2.0,0.0,18.0,0.0,11.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
18178,chr9,9432308,A,T,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,9.0,...,2.0,0.0,16.0,0.0,13.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
18179,chr9,77269794,C,T,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,14.0,14.0,...,2.0,0.0,15.0,0.0,18.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM


In [7]:
# generate and save an RGB image of possible DNMs from the predictions
for row in range(40):
    sample = DNMs.iloc[row]
    
    child = SingleVariant(sample['Chromosome'], int(sample['Start position']), int(sample['Start position']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19521.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
    father = SingleVariant(sample['Chromosome'], int(sample['Start position']), int(sample['Start position']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19514.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
    mother = SingleVariant(sample['Chromosome'], int(sample['Start position']), int(sample['Start position']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19492.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))

    trio = TrioVariant(mother, father, child)
    TrioVariant.save_image(f'/ifs/home/ina/DeNovoCNN/test/images_predicted/lrs_image_pos{sample["Start position"]}.png', trio.image)

In [4]:
# generate and save an RGB image of possible DNMs from the dataset
for row in range(10):
    sample = possible_DNMs.iloc[row]
    
    child = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19521.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
    father = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19514.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
    mother = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19492.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))

    trio = TrioVariantLRS(child, father, mother)
    TrioVariantLRS.save_image(f'/ifs/data/research/projects/ina/ina/DeNovoCNN/test/phasing_encoding_images/phased_image_pos{sample["pos"]}.png', np.flip(trio.image,2))

HP1:  10
HP2:  14
HP3:  0
noHP:  136
HP1:  0
HP2:  1
HP3:  15
noHP:  144
HP1:  6
HP2:  10
HP3:  0
noHP:  144
HP1:  17
HP2:  21
HP3:  0
noHP:  122
HP1:  12
HP2:  8
HP3:  0
noHP:  140
HP1:  14
HP2:  12
HP3:  0
noHP:  134
HP1:  14
HP2:  22
HP3:  0
noHP:  124
HP1:  20
HP2:  17
HP3:  0
noHP:  123
HP1:  20
HP2:  22
HP3:  0
noHP:  118
HP1:  13
HP2:  11
HP3:  0
noHP:  136
HP1:  15
HP2:  21
HP3:  0
noHP:  124
HP1:  14
HP2:  14
HP3:  0
noHP:  132
HP1:  15
HP2:  14
HP3:  0
noHP:  131
HP1:  9
HP2:  18
HP3:  5
noHP:  128
HP1:  26
HP2:  18
HP3:  0
noHP:  116
HP1:  12
HP2:  12
HP3:  0
noHP:  136
HP1:  9
HP2:  9
HP3:  0
noHP:  142
HP1:  6
HP2:  8
HP3:  1
noHP:  145
HP1:  14
HP2:  17
HP3:  0
noHP:  129
HP1:  18
HP2:  18
HP3:  0
noHP:  124
HP1:  20
HP2:  15
HP3:  0
noHP:  125
HP1:  19
HP2:  21
HP3:  0
noHP:  120
HP1:  15
HP2:  14
HP3:  0
noHP:  131
HP1:  10
HP2:  17
HP3:  0
noHP:  133
HP1:  19
HP2:  21
HP3:  0
noHP:  120
HP1:  16
HP2:  14
HP3:  0
noHP:  130
HP1:  11
HP2:  12
HP3:  0
noHP:  137
HP1:  13


In [6]:
# generate images for unknown variants
unknown = dataset[(dataset['DNM_status'] == "UNKNOWN") & (dataset['SAMPLE'] == 'DNA21-19521')]

for row in range(unknown.shape[0]):
    sample = unknown.iloc[row]
    
    child = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19521.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
    father = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19514.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
    mother = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, '/ifs/data/research/projects/ina/test_trio/DNA21-19492.bam', pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))

    trio = TrioVariantLRS(child, father, mother)
    TrioVariantLRS.save_image(f'/ifs/data/research/projects/ina/ina/DeNovoCNN/test/phasing_encoding_images/unknown_phased_image_pos{sample["pos"]}.png', trio.image)

In [7]:
# variants in the child DNA21-19521
child = dataset[dataset['SAMPLE']=='DNA21-19521']
child[80:]

,chrom,pos,ref,alt,hap_counts,variant_type,SAMPLE,number_of_haplotypes,hap1_var_reads,hap1_total_reads,hap2_var_reads,hap2_total_reads,hap_na_var_reads,hap_na_total_reads,hap1_vaf,hap2_vaf,DNM_status
28441,chr6,60938013,G,A,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,15.0,15.0,0.0,21.0,0.0,0.0,1.000000,0.000000,POSSIBLY_PHASED_DNM
28442,chr6,70641248,G,T,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,8.0,8.0,0.0,15.0,0.0,0.0,1.000000,0.000000,POSSIBLY_PHASED_DNM
28443,chr6,78592379,C,T,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,15.0,13.0,13.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
28444,chr6,133296019,A,ATTGC,defaultdict(<function count_variants_by_haplot...,ins,DNA21-19521,2.0,0.0,18.0,16.0,16.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
28445,chr6,145027973,A,C,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,16.0,12.0,12.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
28446,chr6,156149668,C,G,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,15.0,14.0,14.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
28447,chr7,6832671,C,A,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,12.0,12.0,0.0,16.0,0.0,0.0,1.000000,0.000000,POSSIBLY_PHASED_DNM
28448,chr7,77485646,A,G,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,16.0,21.0,21.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
28449,chr7,106935854,C,T,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,16.0,16.0,0.0,15.0,0.0,0.0,1.000000,0.000000,POSSIBLY_PHASED_DNM
28450,chr8,28259843,A,C,defaultdict(<function count_variants_by_haplot...,snp,DNA21-19521,2.0,0.0,13.0,12.0,12.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM


In [8]:
child2 = dataset[dataset['SAMPLE'] == 'DNA24-12831']
child2[40:]

,chrom,pos,ref,alt,hap_counts,variant_type,SAMPLE,number_of_haplotypes,hap1_var_reads,hap1_total_reads,hap2_var_reads,hap2_total_reads,hap_na_var_reads,hap_na_total_reads,hap1_vaf,hap2_vaf,DNM_status
36630,chr2,210994199,C,T,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,18.0,18.0,18.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
36631,chr2,217419792,T,A,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,26.0,18.0,18.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
36632,chr20,49307173,C,A,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,17.0,17.0,0.0,10.0,0.0,0.0,1.000000,0.000000,POSSIBLY_PHASED_DNM
36633,chr20,52289964,T,A,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,14.0,22.0,22.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
36634,chr20,61947594,C,G,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,5.0,12.0,12.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
36635,chr21,24782443,A,G,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,18.0,21.0,21.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
36636,chr21,34872105,A,G,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,17.0,20.0,20.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
36637,chr4,16713751,G,A,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,17.0,15.0,16.0,0.0,0.0,0.000000,0.937500,POSSIBLY_PHASED_DNM
36638,chr4,16713752,C,A,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,16.0,16.0,16.0,0.0,0.0,0.000000,1.000000,POSSIBLY_PHASED_DNM
36639,chr4,45747686,G,C,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,20.0,20.0,0.0,13.0,0.0,0.0,1.000000,0.000000,POSSIBLY_PHASED_DNM


In [5]:
# function to access a specific .bam file based on a dna-id
def get_revio_bam_path(dna_id):

    files_lst = (

        glob.glob(f'/ifs/data/research/revio/work/{dna_id}/GRCh38*/{dna_id}*.bam') + 

        glob.glob(f'/ifs/data/research/revio/work/{dna_id}/GRCh38*/BAMs_*/{dna_id}*.bam')

    )


    if len(files_lst) == 0: raise FileNotFoundError(f"No bam files found for {dna_id}")

    if len(files_lst) > 1: files_lst = sorted(files_lst, key=os.path.getmtime)



    return files_lst[-1]

In [6]:
# Get paths to trio bams
trios = pd.read_csv("/ifs/data/research/revio/work/familyInfo.txt", sep= '\t')
dna_id = 'DNA24-12831'

child = trios.loc[trios['child'] == dna_id, 'child'].iloc[0]
mother = trios.loc[trios['child'] == dna_id, 'father'].iloc[0]
father = trios.loc[trios['child'] == dna_id, 'mother'].iloc[0]

print('child: ' + get_revio_bam_path(child))
print('father: ' + get_revio_bam_path(father))
print('mother: ' + get_revio_bam_path(mother))

child: /ifs/data/research/revio/work/DNA24-12831/GRCh38_20250123/DNA24-12831.bam
father: /ifs/data/research/revio/work/DNA24-12834/GRCh38_20250123/DNA24-12834.bam
mother: /ifs/data/research/revio/work/DNA24-12833/GRCh38_20250123/DNA24-12833.bam


In [9]:
# Read .bam
bam = pysam.AlignmentFile("/ifs/data/research/revio/work/DNA24-12833/GRCh38_20250123/DNA24-12833.bam", "rb")
# first_read = next(bam)
# print(first_read)

has_unphased = False
row = -1

for read in bam:
    row = row + 1
    if not read.has_tag("HP") or read.get_tag("HP") == 0:
        has_unphased = True
        print(row)
        break

bam.close()
print(has_unphased)
print(row)

16
True
16
